# ⚡ Real-Time Streaming CDC Engine (10-Second Polling Stream)

This notebook demonstrates **true automated streaming & Change Data Capture (CDC)** without hidden `.py` black boxes.
All the read/write streaming logic is written **directly inside this notebook** so you can see exactly how it works.

### How It Works:
- **`readStream` (Source)**: Every 10 seconds, the loop reads [`data/stream_landing/stream_source.csv`](file:///c:/Users/sivak/OneDrive/Pictures/blank_backup/Projectx/tech-news-arr-warehouse/data/stream_landing/stream_source.csv).
- **CDC Reconciliation**: Evaluates `op_code`:
  - **`I` (Insert)**: Appends new entities.
  - **`U` (Update)**: Overwrites existing entities in-place (Atomic Upsert).
  - **`D` (Delete)**: Removes retracted entities.
- **`writeStream` (Destination)**: Writes the live state to [`data/stream_landing/stream_destination.csv`](file:///c:/Users/sivak/OneDrive/Pictures/blank_backup/Projectx/tech-news-arr-warehouse/data/stream_landing/stream_destination.csv).
- **Interactive Test**: While the 60-second loop runs, **open `stream_source.csv` in VS Code, add/edit a row, press `Ctrl+S`**, and watch the loop capture it live!

In [5]:
import os
import sys
from pathlib import Path
import time
from datetime import datetime, timezone
import pandas as pd
import duckdb

# Ensure working directory is project root
while not (Path.cwd() / 'part1_tech_news_lakehouse').is_dir() and Path.cwd().parent != Path.cwd():
    os.chdir('..')

# Folder paths
stream_dir = os.path.join(Path.cwd(), 'data', 'stream_landing')
os.makedirs(stream_dir, exist_ok=True)

source_csv = os.path.join(stream_dir, 'stream_source.csv')
dest_csv = os.path.join(stream_dir, 'stream_destination.csv')

# Initialize fresh source file with 2 baseline events
initial_events = pd.DataFrame([
    {
        'event_id': 'EVT_001',
        'event_timestamp': '2026-10-25T10:00:00',
        'op_code': 'I',
        'article_id': 'ART0004',
        'company_name': 'Confluent',
        'arr_usd_M': 980.0,
        'title': 'Confluent Reports Record Revenue Growth'
    },
    {
        'event_id': 'EVT_002',
        'event_timestamp': '2026-10-25T10:05:00',
        'op_code': 'I',
        'article_id': 'ART0006',
        'company_name': 'NVIDIA',
        'arr_usd_M': 16800.0,
        'title': 'NVIDIA Achieves Profitability Milestone'
    }
])
initial_events.to_csv(source_csv, index=False)

# Reset destination file to empty state
pd.DataFrame(columns=['article_id', 'company_name', 'arr_usd_M', 'title', 'last_event_id', 'last_updated_at']).to_csv(dest_csv, index=False)

print("[+] Files Initialized in data/stream_landing/:")
print("    - Source:     ", source_csv)
print("    - Destination:", dest_csv)
print("\nInitial stream_source.csv:")
display(initial_events)


[+] Files Initialized in data/stream_landing/:
    - Source:      c:\Users\sivak\OneDrive\Pictures\blank_backup\Projectx\tech-news-arr-warehouse\data\stream_landing\stream_source.csv
    - Destination: c:\Users\sivak\OneDrive\Pictures\blank_backup\Projectx\tech-news-arr-warehouse\data\stream_landing\stream_destination.csv

Initial stream_source.csv:


,event_id,event_timestamp,op_code,article_id,company_name,arr_usd_M,title
0,EVT_001,2026-10-25T10:00:00,I,ART0004,Confluent,980.0,Confluent Reports Record Revenue Growth
1,EVT_002,2026-10-25T10:05:00,I,ART0006,NVIDIA,16800.0,NVIDIA Achieves Profitability Milestone


## ⚙️ 1. Pure Inline CDC Engine (ReadStream ➔ Reconcile ➔ WriteStream)

Here is the transparent CDC logic written directly in Python/Pandas without any external black-box library:

In [6]:
def run_cdc_micro_batch(src_path: str, dst_path: str):
    """
    1. readStream:  Reads stream_source.csv
    2. CDC Logic:   Reconciles Inserts (I), Updates (U), and Deletes (D)
    3. writeStream: Writes updated state to stream_destination.csv
    """
    if not os.path.exists(src_path) or os.path.getsize(src_path) == 0:
        return False, 0

    # 1. readStream
    df_src = pd.read_csv(src_path, keep_default_na=False)

    # 2. CDC Reconciliation in-memory
    records = {}
    for _, row in df_src.iterrows():
        op = str(row.get('op_code', 'I')).strip().upper()
        art_id = str(row.get('article_id', '')).strip()
        if not art_id:
            continue

        if op == 'D':
            # Delete record
            if art_id in records:
                del records[art_id]
        elif op in ['I', 'U']:
            # Insert or In-Place Update (Atomic Upsert)
            records[art_id] = {
                'article_id': art_id,
                'company_name': row.get('company_name', ''),
                'arr_usd_M': float(row.get('arr_usd_M', 0.0)) if row.get('arr_usd_M') not in ['', None] else 0.0,
                'title': row.get('title', ''),
                'last_event_id': row.get('event_id', ''),
                'last_updated_at': row.get('event_timestamp', '')
            }

    df_new_dest = pd.DataFrame(list(records.values())) if records else pd.DataFrame(
        columns=['article_id', 'company_name', 'arr_usd_M', 'title', 'last_event_id', 'last_updated_at']
    )

    # Check if destination changed compared to disk
    changed = True
    if os.path.exists(dst_path) and os.path.getsize(dst_path) > 0:
        df_old_dest = pd.read_csv(dst_path)
        if df_old_dest.equals(df_new_dest):
            changed = False

    # 3. writeStream
    if changed:
        df_new_dest.to_csv(dst_path, index=False)

    return changed, len(df_new_dest)

print("[+] CDC Ingestion Function defined!")


[+] CDC Ingestion Function defined!


## ⏱️ 2. The 10-Second Live Streaming Loop (Runs for 1 Minute)

Run the cell below! It will poll every 10 seconds for 60 seconds total.

### 🎯 While it is running:
1. Open [`data/stream_landing/stream_source.csv`](file:///c:/Users/sivak/OneDrive/Pictures/blank_backup/Projectx/tech-news-arr-warehouse/data/stream_landing/stream_source.csv) in VS Code.
2. Add a new row to update Confluent to `$1,050M` (`op_code = U`):
   ```csv
   EVT_003,2026-10-25T10:10:00,U,ART0004,Confluent,1050.0,Confluent Q4 Revenue Revised
   ```
3. Or add a row to delete NVIDIA (`op_code = D`):
   ```csv
   EVT_004,2026-10-25T10:15:00,D,ART0006,NVIDIA,16800.0,NVIDIA Article Retracted
   ```
4. Hit **`Ctrl+S` (Save)** in VS Code.
5. Look back at this cell — the loop will detect your change within 10 seconds and update `stream_destination.csv` live!

In [7]:
total_duration_sec = 60
interval_sec = 10
iterations = total_duration_sec // interval_sec

print(f"🚀 Starting Live Stream Polling Loop ({total_duration_sec}s total, waking up every {interval_sec}s)...")
print("👉 Open 'data/stream_landing/stream_source.csv' in VS Code to insert/update rows now!\n")

for tick in range(1, iterations + 1):
    timestamp_str = datetime.now().strftime("%H:%M:%S")
    changed, active_rows = run_cdc_micro_batch(source_csv, dest_csv)

    if changed:
        print(f"[{timestamp_str} | Tick {tick}/{iterations}] ⚡ LIVE CHANGE DETECTED! stream_destination.csv updated ({active_rows} active entities).")
        # Display current destination snapshot
        df_dest_snapshot = pd.read_csv(dest_csv)
        display(df_dest_snapshot[['article_id', 'company_name', 'arr_usd_M', 'last_event_id', 'last_updated_at']])
    else:
        print(f"[{timestamp_str} | Tick {tick}/{iterations}] 💤 No new events in stream_source.csv. Sleeping {interval_sec}s...")

    if tick < iterations:
        time.sleep(interval_sec)

print("\n✅ 1-Minute Live Streaming Session Completed!")


🚀 Starting Live Stream Polling Loop (60s total, waking up every 10s)...
👉 Open 'data/stream_landing/stream_source.csv' in VS Code to insert/update rows now!

[12:58:54 | Tick 1/6] ⚡ LIVE CHANGE DETECTED! stream_destination.csv updated (2 active entities).


,article_id,company_name,arr_usd_M,last_event_id,last_updated_at
0,ART0004,Confluent,980.0,EVT_001,2026-10-25T10:00:00
1,ART0006,NVIDIA,16800.0,EVT_002,2026-10-25T10:05:00


[12:59:04 | Tick 2/6] 💤 No new events in stream_source.csv. Sleeping 10s...
[12:59:14 | Tick 3/6] 💤 No new events in stream_source.csv. Sleeping 10s...
[12:59:24 | Tick 4/6] 💤 No new events in stream_source.csv. Sleeping 10s...
[12:59:34 | Tick 5/6] ⚡ LIVE CHANGE DETECTED! stream_destination.csv updated (2 active entities).


,article_id,company_name,arr_usd_M,last_event_id,last_updated_at
0,ART0004,Confluent,1050.0,EVT_003,2026-10-25T10:10:00
1,ART0006,NVIDIA,16800.0,EVT_002,2026-10-25T10:05:00


[12:59:44 | Tick 6/6] 💤 No new events in stream_source.csv. Sleeping 10s...

✅ 1-Minute Live Streaming Session Completed!


## 📊 3. Inspect Final Destination Table via Pure SQL in DuckDB

Query the destination CSV directly using SQL to verify all live inserts, updates, and deletes:

In [ ]:
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE TABLE live_destination AS SELECT * FROM read_csv_auto('{dest_csv}')")

print("=== Final Reconciled Destination State (DuckDB SQL) ===")
display(con.execute("""
    SELECT article_id, company_name, arr_usd_M, last_event_id, last_updated_at
    FROM live_destination
    ORDER BY arr_usd_M DESC
""").df())
